# Camada Gold — Dimensões + Fato (Delta Live Tables)

Este notebook define as cinco dimensões e a tabela fato do esquema estrela como um pipeline DLT.

**Não execute célula a célula** — associe a um pipeline Delta Live Tables no Databricks
(Workflows → Delta Live Tables → Create Pipeline).

Configure o pipeline com:
- **Pipeline mode:** Triggered
- **Target catalog:** `judicial`
- **Target schema:** `gold`

**Fontes:**
- `judicial.silver.movimentos` → dim_tribunais, dim_classes, dim_orgaos, fato_movimentos
- `judicial.bronze.datajud_raw` → dim_assuntos
- Gerada programaticamente → dim_calendario

**Destino:** `judicial.gold.*`

**Ordem de execução (resolvida automaticamente pelo DLT):**
```
dim_tribunais ──┐
dim_classes   ──┤
dim_assuntos  ──┤──► fato_movimentos
dim_orgaos ◄───┘
dim_calendario──┘
```

In [ ]:
import dlt
from pyspark.sql.functions import (
    sha2, concat_ws, col, explode, lit, sequence,
    to_date, date_format, year, month, quarter, dayofweek,
    when, current_date,
)

## dim_tribunais

Uma linha por tribunal. Surrogate key gerada com `sha2` sobre a sigla.

**Por que `sha2` e não a sigla diretamente como PK?**  
A chave natural (`sigla_tribunal`) é uma string de negócio — pode mudar, pode ter variações de formatação entre fontes. A surrogate key é estável e independente da origem: mesmo que a sigla mude, a FK nas tabelas fato não precisa ser reescrita.

In [ ]:
@dlt.table(
    name="dim_tribunais",
    comment="Dimensão de tribunais — uma linha por sigla",
)
@dlt.expect("sk_tribunal não nula", "sk_tribunal IS NOT NULL")
@dlt.expect("sigla_tribunal não nula", "sigla_tribunal IS NOT NULL")
def dim_tribunais():
    return (
        spark.table("judicial.silver.movimentos")
        .select("sigla_tribunal").distinct()
        .withColumn(
            "sk_tribunal",
            sha2(concat_ws("|", col("sigla_tribunal")), 256)
        )
    )

## dim_classes

Classes processuais CNJ (ex: "Embargos à Execução", "Ação Civil Pública").
Surrogate key gerada sobre o código numérico.

In [ ]:
@dlt.table(
    name="dim_classes",
    comment="Dimensão de classes processuais CNJ",
)
@dlt.expect("sk_classe não nula", "sk_classe IS NOT NULL")
@dlt.expect("codigo_classe não nulo", "codigo_classe IS NOT NULL")
def dim_classes():
    return (
        spark.table("judicial.silver.movimentos")
        .select("codigo_classe", "nome_classe").distinct()
        .withColumn(
            "sk_classe",
            sha2(concat_ws("|", col("codigo_classe").cast("string")), 256)
        )
    )

## dim_assuntos

Lê direto da Bronze porque a Silver só armazena movimentos explodidos —
os assuntos ficam em `_source.assuntos` (array) e não foram propagados para a Silver.

In [ ]:
@dlt.table(
    name="dim_assuntos",
    comment="Dimensão de assuntos processuais CNJ",
)
@dlt.expect("sk_assunto não nula", "sk_assunto IS NOT NULL")
@dlt.expect("codigo_assunto não nulo", "codigo_assunto IS NOT NULL")
def dim_assuntos():
    return (
        spark.table("judicial.bronze.datajud_raw")
        .select(explode(col("_source.assuntos")).alias("assunto"))
        .select(
            col("assunto.codigo").alias("codigo_assunto"),
            col("assunto.nome").alias("nome_assunto"),
        )
        .filter(col("codigo_assunto").isNotNull())
        .distinct()
        .withColumn(
            "sk_assunto",
            sha2(concat_ws("|", col("codigo_assunto").cast("string")), 256)
        )
    )

## dim_orgaos

Órgãos julgadores (varas, câmaras). Referencia `dim_tribunais` via `dlt.read()` —
o DLT detecta essa dependência e garante que `dim_tribunais` é criada antes.

In [ ]:
@dlt.table(
    name="dim_orgaos",
    comment="Dimensão de órgãos julgadores (varas, câmaras)",
)
@dlt.expect("sk_orgao não nula", "sk_orgao IS NOT NULL")
def dim_orgaos():
    # dlt.read() declara dependência explícita com dim_tribunais no mesmo pipeline
    tribunais = dlt.read("dim_tribunais").select("sigla_tribunal", "sk_tribunal")

    return (
        spark.table("judicial.silver.movimentos")
        .select("codigo_orgao", "nome_orgao", "sigla_tribunal").distinct()
        .withColumn(
            "sk_orgao",
            sha2(concat_ws("|", col("codigo_orgao").cast("string")), 256)
        )
        .join(tribunais, on="sigla_tribunal", how="left")
        .withColumnRenamed("sk_tribunal", "fk_tribunal")
        .drop("sigla_tribunal")
    )

## dim_calendario

Gerada programaticamente — não vem de nenhuma fonte de dados.
`sequence()` produz um array com todos os dias de 2020 até hoje;
`explode()` transforma esse array em uma linha por dia.

In [ ]:
@dlt.table(
    name="dim_calendario",
    comment="Dimensão de datas de 2020-01-01 até hoje",
)
@dlt.expect("sk_data não nula", "sk_data IS NOT NULL")
def dim_calendario():
    return (
        spark.range(1)
        .select(
            explode(
                sequence(to_date(lit("2020-01-01")), current_date())
            ).alias("data")
        )
        .withColumn("sk_data",             date_format("data", "yyyyMMdd").cast("int"))
        .withColumn("ano",                 year("data"))
        .withColumn("mes",                 month("data"))
        .withColumn("nome_mes",            date_format("data", "MMMM"))
        .withColumn("trimestre",           quarter("data"))
        .withColumn("semestre",            when(month("data") <= 6, lit(1)).otherwise(lit(2)))
        .withColumn("dia_semana",          date_format("data", "EEEE"))
        # dayofweek: 1=domingo, 7=sábado no Spark
        .withColumn("is_fim_de_semana",    dayofweek("data").isin(1, 7))
        .withColumn("is_feriado_nacional", lit(False))
    )

---

## fato_movimentos

Cada linha é um evento processual (movimento). Armazena apenas FKs e métricas —
nenhum atributo descritivo. Para saber o nome do tribunal, faça JOIN com `dim_tribunais`.

**FKs para datas:** em vez de fazer JOIN com `dim_calendario`, calculamos `sk_data`
diretamente com `date_format(..., "yyyyMMdd").cast("int")` — o resultado é idêntico
ao `sk_data` da dimensão, sem o custo de um JOIN extra.

**`tipo_fase`:** mapeamento de `codigo_movimento` para fase analítica, conforme
tabela CNJ de movimentos (ranges de código por fase).

In [ ]:
@dlt.table(
    name="fato_movimentos",
    comment="Fato de movimentos processuais — uma linha por evento",
)
@dlt.expect("fk_tribunal não nula",       "fk_tribunal IS NOT NULL")
@dlt.expect("fk_classe não nula",         "fk_classe IS NOT NULL")
@dlt.expect("fk_data_movimento não nula", "fk_data_movimento IS NOT NULL")
def fato_movimentos():
    silver     = spark.table("judicial.silver.movimentos")
    tribunais  = dlt.read("dim_tribunais").select("sigla_tribunal", col("sk_tribunal").alias("fk_tribunal"))
    classes    = dlt.read("dim_classes").select("codigo_classe",   col("sk_classe").alias("fk_classe"))
    orgaos     = dlt.read("dim_orgaos").select("codigo_orgao",     col("sk_orgao").alias("fk_orgao"))

    # sk_data = inteiro yyyyMMdd — mesmo formato do dim_calendario.sk_data, sem JOIN
    tipo_fase = (
        when((col("codigo_movimento") >= 1)   & (col("codigo_movimento") < 100), "distribuição")
        .when((col("codigo_movimento") >= 100) & (col("codigo_movimento") < 200), "conhecimento")
        .when((col("codigo_movimento") >= 200) & (col("codigo_movimento") < 300), "instrução")
        .when((col("codigo_movimento") >= 300) & (col("codigo_movimento") < 400), "sentença")
        .when((col("codigo_movimento") >= 400) & (col("codigo_movimento") < 500), "recurso")
        .when((col("codigo_movimento") >= 500) & (col("codigo_movimento") < 600), "execução")
        .when((col("codigo_movimento") >= 600) & (col("codigo_movimento") < 700), "arquivamento")
        .otherwise("outro")
    )

    return (
        silver
        .join(tribunais, on="sigla_tribunal", how="left")
        .join(classes,   on="codigo_classe",  how="left")
        .join(orgaos,    on="codigo_orgao",   how="left")
        .select(
            # chave natural do processo (mantida para rastreabilidade)
            "numero_processo",
            # chaves estrangeiras para dimensões
            "fk_tribunal",
            "fk_classe",
            "fk_orgao",
            date_format(col("data_movimento").cast("date"),  "yyyyMMdd").cast("int").alias("fk_data_movimento"),
            date_format(col("data_ajuizamento").cast("date"),"yyyyMMdd").cast("int").alias("fk_data_ajuizamento"),
            # atributos degenerados — códigos sem dimensão própria
            "codigo_movimento",
            "grau",
            tipo_fase.alias("tipo_fase"),
            # medidas
            "dias_desde_ajuizamento",
            "dias_desde_ultimo_movimento",
            "is_ultimo_movimento",
        )
    )